# Can we predict 30-day readmission?
Target: `readmit_30d`. The data has 11,868 stays from 1,500 people (~8 stays each), so **rows are not independent**. All validation below is grouped by `person_key`, so the same person never appears in both train and test.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); df.head()

## 1. Data checks

In [ ]:
print(df.shape, '| people:', df.person_key.nunique(), '| missing:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum())
print('Readmission rate:', df.readmit_30d.mean().round(3))
g_=df.groupby('person_key')
print('Persons with >1 birth_year:', (g_.birth_year.nunique()>1).sum(), '| >1 sex:', (g_.sex.nunique()>1).sum(), '| >1 postcode:', (g_.postcode_area.nunique()>1).sum())
print('Duplicate (person, visit_number):', df.duplicated(['person_key','visit_number']).sum())
df.describe().T

## 2. Exploration: univariate relationships with the outcome

In [ ]:
num=['birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b','visit_number']
fig,ax=plt.subplots(2,4,figsize=(16,7)); ax=ax.ravel()
for a,c in zip(ax,num):
    q=pd.qcut(df[c],10,duplicates='drop'); r=df.groupby(q,observed=True).readmit_30d.mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(df.readmit_30d.mean(),ls='--',c='gray'); a.set_title(c+' (decile)'); a.set_ylabel('readmit rate')
ax[-1].axis('off'); plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean().round(3))

In [ ]:
print(df[num].corrwith(df.readmit_30d).round(3).sort_values())
print('postcode areas:', df.postcode_area.nunique(), '| chi-square p =', chi2_contingency(pd.crosstab(df.postcode_area,df.readmit_30d))[1].round(4))

## 3. Modelling setup
- `person_key` is an identifier, not a feature. `postcode_area` is a code, so it is one-hot encoded (logistic) / treated as categorical (boosting).
- 5-fold `StratifiedGroupKFold` by person; predictions are out-of-fold. Metrics: ROC-AUC (ranking) and PR-AUC (baseline = prevalence, about 0.20).

In [ ]:
feats=['visit_number','birth_year','sex','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
X=df[feats].copy(); y=df.readmit_30d.values; g=df.person_key.values
cv=StratifiedGroupKFold(5,shuffle=True,random_state=0)
numc=[c for c in feats if c not in('sex','postcode_area')]
def make_lin(): return make_pipeline(ColumnTransformer([('n',StandardScaler(),numc),('c',OneHotEncoder(handle_unknown='ignore'),['sex','postcode_area'])]),LogisticRegression(max_iter=2000))
lin=make_lin()
Xg=X.assign(sex=(X.sex=='F').astype(int))
gb=HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,categorical_features=[feats.index('postcode_area')],random_state=0)
models={'Baseline (prevalence)':(DummyClassifier(),X),'Logistic regression':(lin,X),'Gradient boosting':(gb,Xg)}
oof={}; rows=[]
for n,(m,Xm) in models.items():
    p=cross_val_predict(m,Xm,y,groups=g,cv=cv,method='predict_proba')[:,1]; oof[n]=p
    fold=[roc_auc_score(y[t],p[t]) for _,t in cv.split(Xm,y,g)]
    rows.append(dict(model=n,ROC_AUC=roc_auc_score(y,p),AUC_fold_sd=np.std(fold),PR_AUC=average_precision_score(y,p),Brier=brier_score_loss(y,p)))
pd.DataFrame(rows).round(4)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
for n,p in oof.items():
    f,t,_=roc_curve(y,p); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(y,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(xlabel='FPR',ylabel='TPR',title='ROC (out-of-fold, grouped)')
ax[1].axhline(y.mean(),c='k',ls=':'); ax[1].set(xlabel='Recall',ylabel='Precision',title='Precision-recall'); ax[0].legend(); plt.tight_layout(); plt.show()

## 4. Does ignoring the grouping inflate results? (leakage check)

In [ ]:
for n in ['Logistic regression','Gradient boosting']:
    m,Xm=models[n]
    naive=cross_val_score(m,Xm,y,cv=StratifiedKFold(5,shuffle=True,random_state=0),scoring='roc_auc').mean()
    print(f'{n}: naive row-wise CV AUC={naive:.4f}  vs grouped CV AUC={roc_auc_score(y,oof[n]):.4f}')

## 5. What drives the predictions?

In [ ]:
m=make_lin().fit(X,y)
names=m[0].get_feature_names_out(); coef=pd.Series(m[-1].coef_[0],names)
print('Standardised logistic coefficients (numeric features, log-odds per 1 SD):')
print(coef[[n for n in names if n.startswith('n__')]].sort_values().round(3))
imp=[]
for tr,te in cv.split(X,y,g):
    mm=make_lin().fit(X.iloc[tr],y[tr])
    imp.append(permutation_importance(mm,X.iloc[te],y[te],scoring='roc_auc',n_repeats=10,random_state=0).importances_mean)
pi=pd.Series(np.mean(imp,0),feats).sort_values(); pi.plot.barh(title='Permutation importance (held-out AUC drop, logistic)'); plt.show()

## 6. Calibration and a practical operating point

In [ ]:
p=oof['Logistic regression']
fp,mp=calibration_curve(y,p,n_bins=10,strategy='quantile'); plt.plot(mp,fp,'o-'); plt.plot([0,.6],[0,.6],'k:'); plt.xlabel('predicted'); plt.ylabel('observed'); plt.title('Calibration (out-of-fold)'); plt.show()
d=pd.DataFrame({'p':p,'y':y}); d['decile']=pd.qcut(d.p.rank(method='first'),10,labels=False)+1
print(d.groupby('decile').y.mean().round(3).to_frame('readmit rate').T)
top=d.p>=d.p.quantile(.8); print(f'Top 20% flagged: precision={d.y[top].mean():.3f} (base {y.mean():.3f}), recall={d.y[top].sum()/y.sum():.3f}')

## 7. Conclusion (numbers from a run of the cells above)
- **Signal exists but is weak.** Grouped out-of-fold ROC-AUC is about 0.62 (logistic regression) vs 0.50 for the baseline; PR-AUC about 0.29 vs 0.20 prevalence. Boosting (about 0.60) does not beat the simple linear model.
- **Main drivers:** longer stay, higher `lab_a`, more medications, and residence area (readmission rate ranges from about 9% to 31% across the 50 postcode areas). Age, sex, BMI, `lab_b` and `visit_number` add essentially nothing.
- **Usable only for coarse risk ranking:** the top-risk decile is readmitted about 36% of the time vs about 11% in the lowest. Flagging the top 20% gives about 31% precision and about 31% recall, so it is not suitable for individual decisions.
- **Leakage warning:** naive row-wise CV overstates AUC (0.66 / 0.69) because the same person appears in train and test; person-grouped CV is the honest estimate.
- Caveats: one-time split by person, not by time; no external validation; person-level readmission rates vary more than chance, so unmeasured person-level factors likely matter.